In [18]:
!python -V

Python 3.11.16


In [19]:
import pandas as pd

In [20]:
import numpy as np

In [21]:
import pickle

In [22]:
import seaborn as sns
import matplotlib.pyplot as plt

In [23]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge

from sklearn.metrics import mean_squared_error

In [24]:
import mlflow


mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("nyc-taxi-experiment")

<Experiment: artifact_location='/workspaces/mlops-zoomcamp/02-experiment-tracking/mlruns/1', creation_time=1788898117863, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1788898117863, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}, trace_location=None, workspace='default'>

In [25]:
import os
print(os.getcwd())
print(mlflow.get_tracking_uri())

/workspaces/mlops-zoomcamp/02-experiment-tracking
sqlite:///mlflow.db


In [26]:
def read_dataframe(filename):
    df = pd.read_csv(filename)

    #On parse les colonne lpep_dropff_date_time. Note : df.lpep_dropoff_datetime est équivalent à faire df['lpep_dropoff_datetime']
    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)

    #On créé un nouvelle colonne 'duration'
    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    #Convert city zones identifier (ex:130,65) into text instead of float/integers, otherwise oython will say 130 is greater than 65, which makes no sense
    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df

In [27]:
df_train = read_dataframe('./data/green_tripdata_2021-01.csv')
df_val = read_dataframe('./data/green_tripdata_2021-02.csv')

/tmp/ipykernel_36907/1374236984.py:2: DtypeWarning: Columns (0: store_and_fwd_flag) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filename)


In [28]:
df_train.head()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,duration
0,2.0,2021-01-01 00:15:56,2021-01-01 00:19:52,N,1.0,43,151,1.0,1.01,5.5,...,0.5,0.00,0.0,NaN,0.3,6.80,2.0,1.0,0.00,3.933333
1,2.0,2021-01-01 00:25:59,2021-01-01 00:34:44,N,1.0,166,239,1.0,2.53,10.0,...,0.5,2.81,0.0,NaN,0.3,16.86,1.0,1.0,2.75,8.750000
2,2.0,2021-01-01 00:45:57,2021-01-01 00:51:55,N,1.0,41,42,1.0,1.12,6.0,...,0.5,1.00,0.0,NaN,0.3,8.30,1.0,1.0,0.00,5.966667
3,2.0,2020-12-31 23:57:51,2021-01-01 00:04:56,N,1.0,168,75,1.0,1.99,8.0,...,0.5,0.00,0.0,NaN,0.3,9.30,2.0,1.0,0.00,7.083333
7,2.0,2021-01-01 00:26:31,2021-01-01 00:28:50,N,1.0,75,75,6.0,0.45,3.5,...,0.5,0.96,0.0,NaN,0.3,5.76,1.0,1.0,0.00,2.316667


In [29]:
len(df_train), len(df_val)

(73908, 61921)

In [30]:
df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
df_val['PU_DO'] = df_val['PULocationID'] + '_' + df_val['DOLocationID']

# PU et DO sont des identifiant de zones administratives (zone 130 = Upper East Side, zone 85  = Greenwich Village, zone 41  = JFK Airport) --> ce sont des valeurs discrètes, poas continues

**DictVectorizer**

C'est une **classe** de la librairie `scikit-learn`. Quand tu écris `dv = DictVectorizer()`, tu crées une **instance** de cette classe. C'est comme un moule dont tu crées un exemplaire concret.

```python
from sklearn.feature_extraction import DictVectorizer
dv = DictVectorizer()  # on crée un objet "vectoriseur"
```

---

**Tes hypothèses sur train_dicts — tu as presque tout bon !**

**`categorical` et `numerical` contiennent juste les noms des colonnes**

```python
categorical = ['PU_DO']       # → juste le nom, une liste de strings
numerical = ['trip_distance'] # → pareil
```
Pas de données dedans, juste des étiquettes. C'est utile pour s'en souvenir et les réutiliser proprement.

---

**`df_train[categorical + numerical]` — oui, on sélectionne les colonnes**

```python
categorical + numerical  # → ['PU_DO', 'trip_distance']

df_train[['PU_DO', 'trip_distance']]  # c'est exactement ce que ça donne
```
On extrait un sous-DataFrame avec uniquement ces deux colonnes.

---

**`.to_dict(orient='records')` — la clé et la valeur**

La **clé** = le nom de la colonne, la **valeur** = la valeur de la cellule :

```python
# Si df_train ressemble à ça :
# PU_DO   | trip_distance
# '130_85' |     2.3
# '41_74'  |     5.1

train_dicts = [
  {'PU_DO': '130_85', 'trip_distance': 2.3},  # ligne 1
  {'PU_DO': '41_74',  'trip_distance': 5.1},  # ligne 2
]
```
Chaque ligne du DataFrame devient un dictionnaire.

**Pourquoi `orient='records'` ?** Parce que `.to_dict()` peut produire plusieurs formats différents :

```python
# orient='records' → une liste de dicts (un par ligne) ✅
[{'PU_DO': '130_85', 'trip_distance': 2.3}, ...]

# orient='dict' (défaut) → un dict de dicts (une entrée par colonne) ❌
{'PU_DO': {0: '130_85', 1: '41_74'}, 'trip_distance': {0: 2.3, 1: 5.1}}

# orient='list' → un dict de listes ❌
{'PU_DO': ['130_85', '41_74'], 'trip_distance': [2.3, 5.1]}
```

`DictVectorizer` attend le format `records` — un dictionnaire par observation — donc on doit le préciser explicitement.


DictVectoiser transform une liste de dictionnaire en des matrice one hot encoding

X_train :
```python
col_130_85 | col_41_74 | col_22_53 | trip_distance
    1      |     0     |     0     |     2.3        ← trajet zone 130_85
    0      |     1     |     0     |     5.1        ← trajet zone 41_74
    0      |     0     |     1     |     1.8        ← trajet zone 22_53
```

X_val :
```python
col_130_85 | col_41_74 | col_22_53 | trip_distance
    0      |     1     |     0     |     3.2        ← trajet zone 41_74
    1      |     0     |     0     |     7.5        ← trajet zone 130_85
    0      |     1     |     0     |     2.1        ← trajet zone 41_74
```




X_val contient moins d'exemple que X_train --> la matrice correspondante va avoir des lignes avec que des zeros 

Exemple : Si la zone 130_85 n'apparaît jamais dans la validation :

X_val :
```python
col_130_85 | col_41_74 | col_22_53 | trip_distance
    0      |     1     |     0     |     3.2    
    0      |     0     |     1     |     7.5    
    0      |     1     |     0     |     2.1    
  ↑
cette colonne existe bien, mais elle vaut 0 partout
```

La colonne col_130_85 est bien présente (pour garder la compatibilité), mais remplie de 0 puisqu'aucun trajet de la validation ne part de cette zone.

Résumé final clair

```
X_train	X_val
Nombre de colonnes	N	N (identique)
Nombre de lignes	Grand	Plus petit
Colonnes absentes	Non	Non, mais peuvent valoir 0 partout
```

In [31]:
categorical = ['PU_DO'] #'PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer() #DictVectoriser est une classe. Ici on initilise l'objet dv de class dictvectorizer

# Ci dessous : on prends les 2 colonnes categorical et numerical (PU_DO et trip_distace) des dataframe. On les transforme en liste de dictionnaire [{'PU_DO':13_42 , 'trip_distance':34},{'PU_DO':13_42 , 'trip_distance':34},... ] car c'est ce que Dictvectorizer attend
# X_train = dv.fit_transform(train_dicts) --> crée une matrice one hot encoding : n+1 colonnes, avec n, le nombre de valeur de PU_DO différente, la dernière colonne étant 'trip disance'. La dernière colonne contient la valeur de trip distance, tandis que els n colonne contienne 0 ou 1 (1 si la PU_DO coorrespondant à cette valeur de trip distance, 0 sinon)
#X_val et X train doivent contenir les même colonnes (comme pour le cours de Andrew NG., avec les sequencial model, on doit avoir les même mots dans le dictionnaire) --> on utilie la méthode .fit_transform pour X_train, et ensuite .transform pour X_val --> ça assure que X_val a les même colonne que X train
# X_train et X_val ont exactement les même n+1 colonne (les même n colonnes correspondant aux valeurs de PU_D0). Mais comme, X°val a moins d'exemple, il aura des lignes avec uniquement des 0 !!
train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [32]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [33]:
# j'ai ajouté ce code pour convertir les int64 en int32 sinon ça faisait un bug
def to_int32(X):
    X = X.tocsr()
    X.indices = X.indices.astype(np.int32)
    X.indptr = X.indptr.astype(np.int32)
    return X

X_train = to_int32(X_train)
X_val = to_int32(X_val)

In [34]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

#mean_squared_error(y_val, y_pred, squared=False) #ne foinctionne pas avec scikitlearn 1.4
np.sqrt(mean_squared_error(y_val, y_pred))

np.float64(7.75871520603533)

**Ce code sauvegarde le modèle entraîné sur le disque.**

---

**`pickle`** est une librairie Python qui permet de **sérialiser** des objets Python — c'est-à-dire les convertir en une suite d'octets pour les stocker dans un fichier. Et de les recharger plus tard à l'identique.

Analogie : c'est comme une **photocopieuse pour objets Python**. Tu "photocopies" l'objet dans un fichier, et tu peux le "réimprimer" plus tard.

---

**Décortiquons ligne par ligne**

```python
with open('models/lin_reg.bin', 'wb') as f_out:
```

- `open('models/lin_reg.bin', 'wb')` → ouvre (ou crée) un fichier nommé `lin_reg.bin` dans le dossier `models/`
- `'wb'` → mode d'ouverture : **w**rite (écriture) + **b**inary (binaire), car pickle écrit des octets, pas du texte
- `with ... as f_out` → ouvre le fichier et le nomme `f_out`. Le `with` garantit que le fichier sera **automatiquement fermé** à la fin du bloc, même si une erreur survient

```python
    pickle.dump((dv, lr), f_out)
```

- `(dv, lr)` → on sauvegarde un **tuple** contenant deux objets :
  - `dv` : le DictVectorizer (avec son vocabulaire appris)
  - `lr` : le modèle LinearRegression (avec ses coefficients appris)
- `f_out` → on écrit dans le fichier ouvert juste avant

---

**Pourquoi sauvegarder les deux ensemble ?**

Parce qu'ils forment un **pipeline indissociable** :

```
données brutes → DictVectorizer → matrice numérique → LinearRegression → prédiction
```

Pour faire une prédiction sur de nouvelles données, tu auras besoin des deux. Les sauvegarder ensemble évite de les dissocier par erreur.

---

**Pour recharger plus tard**

```python
with open('models/lin_reg.bin', 'rb') as f_in:  # 'rb' = read binary
    dv, lr = pickle.load(f_in)
```

Tu retrouves exactement `dv` et `lr` dans l'état où tu les avais laissés, prêts à faire des prédictions.

In [35]:
#On uvre (ou crée) un fichier nommé lin_reg.bin dans le dossier models/. 
# 'wb' → mode d'ouverture : write (écriture) + binary (binaire), car pickle écrit des octets, pas du texte
# with ... as f_out → ouvre le fichier et le nomme f_out. Le with garantit que le fichier sera automatiquement fermé à la fin du bloc, même si une erreur survient
#(dv, lr) → on sauvegarde un tuple contenant deux objets :
#dv : le DictVectorizer (avec son vocabulaire appris)
#lr : le modèle LinearRegression (avec ses coefficients appris)
#f_out → on écrit dans le fichier ouvert juste avant

with open('models/lin_reg.bin', 'wb') as f_out:
    pickle.dump((dv, lr), f_out)

Ta compréhension est très bonne dans l'ensemble ! Je te confirme point par point et je corrige deux choses importantes.

---

**Ce qui est juste ✅**

- `mlflow.set_tracking_uri` + `set_experiment` → oui, exécutés une seule fois avant. `set_experiment` crée l'experiment s'il n'existe pas, sinon il le réutilise.
- `with mlflow.start_run()` → oui, chaque bloc `with` = un run dans cet experiment.
- Les tags → oui, métadonnées libres pour retrouver/filtrer les runs plus tard. Autant que tu veux.
- Les params → oui, les entrées du run (chemins des données, hyperparamètres).
- La logique entraînement → prédiction → mesure de l'erreur → oui.

---

**Correction 1 : ce n'est pas une régression linéaire simple**

```python
lr = Lasso(alpha)
```

C'est un **Lasso**, pas une `LinearRegression`. C'est une régression linéaire **avec régularisation L1** : elle pénalise les gros coefficients, ce qui force certains à devenir exactement 0. Utile ici car avec le one-hot encoding on a des milliers de colonnes.

Le nom de variable `lr` est trompeur (héritage du copier-coller de la cellule précédente), mais c'est bien un modèle différent.

`alpha` est l'hyperparamètre qui contrôle la force de cette pénalité — d'où l'intérêt de le logger : tu pourras comparer alpha=0.1, alpha=0.01, etc.

---

**Correction 2 : params vs tags, la nuance**

| | Usage |
|---|---|
| `log_param` | Ce qui **définit** le run (hyperparamètres, données d'entrée) |
| `set_tag` | Métadonnées d'**organisation** (qui, quoi, contexte) |

En pratique MLflow te laisse mettre ce que tu veux où tu veux, mais la convention est utile : les params sont ce que tu fais varier pour comparer, les tags servent à filtrer.

---

**Petite nuance sur `rmse`**

Tu dis "fonction de coût". Attention, ce sont deux choses distinctes ici :
- La fonction de coût **optimisée pendant l'entraînement** = MSE + pénalité L1
- Le RMSE = la **métrique d'évaluation** que tu regardes pour juger le modèle

Ça se ressemble mais ce n'est pas identique.

---

**La ligne que tu as oubliée de mentionner**

```python
mlflow.log_artifact(local_path="models/lin_reg.bin", artifact_path="models_pickle")
```

Elle enregistre le **fichier pickle** dans MLflow, en plus des métriques. Un artifact = n'importe quel fichier associé au run.

⚠️ Attention : ce fichier `lin_reg.bin` a été créé plus tôt avec la `LinearRegression`, **pas avec le Lasso** de ce run. Donc tu enregistres l'ancien modèle. C'est une incohérence du notebook du cours. Pour bien faire, il faudrait re-sauvegarder le pickle avec le Lasso avant de le logger.

---

**Ta question : pourquoi réentraîner ?**

Deux raisons, et la deuxième est la plus importante.

**1. Ce n'est pas le même modèle**
Cellule 1 = `LinearRegression`, cellule 2 = `Lasso`. Ce sont deux expériences différentes qu'on veut comparer.

**2. MLflow ne peut tracker que ce qui se passe DANS le bloc `with`**

C'est le cœur du module 2. MLflow n'a aucune visibilité sur ce que tu as fait avant :

```python
# Hors du with → MLflow ne voit rien
lr.fit(X_train, y_train)

with mlflow.start_run():
    lr.fit(X_train, y_train)  # là MLflow peut enregistrer le contexte
```

Le premier entraînement était juste un **baseline exploratoire** à la main. Le deuxième est la même démarche mais **instrumentée** : tout est enregistré, comparable, reproductible. C'est précisément le passage du "je bricole dans un notebook" au "je trace mes expériences proprement".

---

**Un dernier point pratique**

```python
rmse = mean_squared_error(y_val, y_pred, squared=False)
```

Cette ligne va planter chez toi (scikit-learn 1.4+). Remplace-la par :

```python
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
```


**Oui, alpha est bien le paramètre de régularisation, pas le learning rate.**

Et il y a une raison plus profonde : **le Lasso de scikit-learn n'a pas de learning rate du tout**. Il n'est pas résolu par descente de gradient, mais par une méthode appelée *coordinate descent* qui n'a pas besoin de pas d'apprentissage.

C'est un piège classique parce que `alpha` désigne souvent le learning rate dans les cours théoriques (notamment chez Andrew Ng, où α est le pas de la descente de gradient). Dans scikit-learn, la convention est inversée : `alpha` = force de régularisation.

---

**La fonction de coût du Lasso**

Voici ce que scikit-learn minimise exactement :

```
(1 / (2n)) × ||y - Xw||²  +  alpha × ||w||₁
   ↑                             ↑
   MSE (erreur quadratique)      pénalité L1
```

- Le premier terme veut **coller aux données**
- Le second veut **garder les coefficients petits**
- `alpha` arbitre entre les deux : alpha grand → priorité à la simplicité, alpha petit → priorité à la précision

Avec alpha = 0, tu retombes exactement sur une `LinearRegression` classique.

---

**Comment le savoir en pratique ?**

**1. Dans Jupyter, le plus rapide :**
```python
Lasso?
```
Un point d'interrogation après le nom ouvre la docstring, qui contient la formule de l'objectif minimisé. Ça marche pour n'importe quelle classe sklearn.

**2. La documentation officielle**

La page de chaque modèle sklearn affiche explicitement l'objectif mathématique en haut de la description.

**3. Le nom du modèle suffit souvent**

C'est une convention stable dans sklearn :

| Modèle | Fonction de coût |
|---|---|
| `LinearRegression` | MSE seul |
| `Ridge` | MSE + alpha × L2 (somme des carrés) |
| `Lasso` | MSE + alpha × L1 (somme des valeurs absolues) |
| `ElasticNet` | MSE + mélange L1 et L2 |

---

**Attention pour la suite du module 2**

Tu vas rencontrer XGBoost, et là `learning_rate` existe vraiment (parfois noté `eta`), en plus de paramètres de régularisation séparés (`reg_alpha`, `reg_lambda`). Donc la règle générale :

- Si le modèle est entraîné par **descente de gradient** (réseaux de neurones, XGBoost, `SGDRegressor`) → il y a un learning rate, et il porte presque toujours ce nom explicitement
- Si le paramètre s'appelle `alpha` dans un modèle linéaire sklearn → c'est de la régularisation

**Tu as tout à fait raison, et ta remarque est très pertinente.**

Le `b` existe bien. Ce que j'ai écrit était une notation raccourcie, un peu paresseuse.

---

**La formule complète est bien celle-ci**

```
(1 / (2n)) × ||y - (Xw + b)||²  +  alpha × ||w||₁
```

Le `b` (appelé *intercept* en anglais, ou biais) est bien présent dans le modèle. Par défaut `Lasso(fit_intercept=True)`, donc scikit-learn l'apprend automatiquement. Tu peux même y accéder après entraînement :

```python
lr.coef_       # les w
lr.intercept_  # le b
```

---

**Pourquoi la formule est souvent écrite sans b — deux raisons**

**1. L'astuce du "vecteur augmenté"**

On peut ajouter une colonne de 1 à la matrice X et faire de b un coefficient comme les autres :

```
X = [x₁, x₂, ..., 1]        w = [w₁, w₂, ..., b]

Alors  Xw  =  w₁x₁ + w₂x₂ + ... + b × 1
                                      ↑
                              c'est exactement le b
```

Avec cette convention, `Xw` contient déjà le biais et la notation est plus compacte. C'est très courant dans les articles et les cours.

**2. Point important : b n'est PAS pénalisé**

Et c'est là le détail crucial. Regarde bien la formule :

```
(1 / (2n)) × ||y - (Xw + b)||²  +  alpha × ||w||₁
                          ↑                  ↑
                    b est présent ici    mais PAS ici
```

Le terme de régularisation `||w||₁` ne contient **que les w**, jamais le b.

---

**Pourquoi ne pas régulariser b ?**

Parce que `b` représente la **valeur moyenne de base** de la prédiction, pas une sensibilité à une feature.

Concrètement dans ton cas : si tous les trajets durent en moyenne 15 minutes, `b` vaudra environ 15. Le pénaliser reviendrait à dire *"essaie de prédire des durées proches de 0"*, ce qui n'a aucun sens.

La régularisation sert à limiter la **complexité** du modèle — c'est-à-dire à quel point il réagit fortement à chaque feature. Le niveau de référence global n'est pas de la complexité.

Autre façon de voir : si tu convertissais tes durées de minutes en secondes, `b` serait multiplié par 60. Une pénalité sur `b` rendrait donc le modèle dépendant de l'unité choisie, ce qui serait absurde.

---

**Pour vérifier toi-même**

```python
from sklearn.linear_model import Lasso
Lasso?
```

La docstring affiche l'objectif exact minimisé. Et tu verras que `fit_intercept` est un paramètre séparé, justement parce que le biais a un traitement à part.

In [36]:

#To give a name to the run, you can follow these 2 methods
#with mlflow.start_run(run_name="lasso-alpha-0.1"):
   # mlflow.set_tag("developer", "cristian")
    #...
#OR
#with mlflow.start_run():
#    mlflow.set_tag("mlflow.runName", "lasso-alpha-0.1")


with mlflow.start_run():

    

    mlflow.set_tag("developer", "cristian")

    mlflow.log_param("train-data-path", "./data/green_tripdata_2021-01.csv")
    mlflow.log_param("valid-data-path", "./data/green_tripdata_2021-02.csv")

    #C'est pas une régression linéaire mais un Lasso, pas une LinearRegression. C'est une régression linéaire avec régularisation L1
    #alpha est ici le paramètr ed erégularsation (le lambda de Andrew Ng., pas le learning rate)
    alpha = 0.1
    mlflow.log_param("alpha", alpha)
    lr = Lasso(alpha)
    lr.fit(X_train, y_train)

    y_pred = lr.predict(X_val)
    #rmse = mean_squared_error(y_val, y_pred, squared=False)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mlflow.log_metric("rmse", rmse)

    mlflow.log_artifact(local_path="models/lin_reg.bin", artifact_path="models_pickle")

In [37]:
print("tracking uri :", mlflow.get_tracking_uri())

run = mlflow.last_active_run()
print("run_id       :", run.info.run_id)
print("experiment_id:", run.info.experiment_id)
print("status       :", run.info.status)
print("artifact_uri :", run.info.artifact_uri)

tracking uri : sqlite:///mlflow.db
run_id       : 507973843ce847da8ee96445395d8b92
experiment_id: 1
status       : FINISHED
artifact_uri : /workspaces/mlops-zoomcamp/02-experiment-tracking/mlruns/1/507973843ce847da8ee96445395d8b92/artifacts


In [38]:
Lasso?

Init signature:
Lasso(
    alpha=1.0,
    *,
    fit_intercept=True,
    precompute=False,
    copy_X=True,
    max_iter=1000,
    tol=0.0001,
    warm_start=False,
    positive=False,
    random_state=None,
    selection='cyclic',
)
Docstring:     
Linear Model trained with L1 prior as regularizer (aka the Lasso).

The optimization objective for Lasso is::

    (1 / (2 * n_samples)) * ||y - Xw||^2_2 + alpha * ||w||_1

Technically the Lasso model is optimizing the same objective function as
the Elastic Net with ``l1_ratio=1.0`` (no L2 penalty).

Read more in the :ref:`User Guide <lasso>`.

Parameters
----------
alpha : float, default=1.0
    Constant that multiplies the L1 term, controlling regularization
    strength. `alpha` must be a non-negative float i.e. in `[0, inf)`.

    When `alpha = 0`, the objective is equivalent to ordinary least
    squares, solved by the :class:`LinearRegression` object. For numerical
    reasons, using `alpha = 0` with the `Lasso` object is not advised.

In [39]:
import xgboost as xgb

In [40]:
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

**Ce que font ces lignes**

Elles convertissent tes données dans le format interne de XGBoost, appelé `DMatrix`.

```python
train = xgb.DMatrix(X_train, label=y_train)
```

Remarque un point important : `X_train` (les features) et `y_train` (la cible) sont **regroupés dans un seul objet**. Le paramètre `label=` sert à préciser lequel est la cible.

---

**Pourquoi on ne peut pas utiliser X_train directement ?**

La réponse courte : on **pourrait**, mais pas avec l'API utilisée dans le cours.

XGBoost propose deux interfaces différentes :

| API | Format attendu | Syntaxe |
|---|---|---|
| **Scikit-learn** | `X_train`, `y_train` normaux | `XGBRegressor().fit(X_train, y_train)` |
| **Native** | `DMatrix` obligatoire | `xgb.train(params, train, ...)` |

Le cours utilise l'API native (`xgb.train`), donc le `DMatrix` est obligatoire. C'est un choix, pas une contrainte absolue.

---

**Pourquoi le cours choisit l'API native ?**

Parce qu'elle permet l'**early stopping** proprement, ce qui sera utilisé juste après :

```python
xgb.train(
    params=params,
    dtrain=train,
    num_boost_round=1000,
    evals=[(valid, 'validation')],      # ← on surveille la validation
    early_stopping_rounds=50            # ← on arrête si ça ne s'améliore plus
)
```

C'est ici que le regroupement features + labels devient utile : tu passes **un seul objet** `valid`, et XGBoost sait tout seul calculer l'erreur à chaque itération. Sans ça, il faudrait passer les deux séparément à chaque fois.

---

**Et techniquement, à quoi sert ce format ?**

Le `DMatrix` n'est pas juste un emballage cosmétique. Il fait un **pré-calcul fait une fois pour toutes** :

- Il range les valeurs de chaque feature dans des "paniers" (histogrammes) que l'algorithme utilise pour chercher les meilleurs seuils de découpe des arbres
- Il compresse la matrice creuse — utile ici, vu que ton one-hot encoding produit des milliers de colonnes presque toutes à 0
- Il gère nativement les valeurs manquantes

Comme XGBoost construit des centaines d'arbres successifs sur les **mêmes données**, faire ce travail une seule fois au départ plutôt qu'à chaque arbre représente un gain de vitesse considérable.

---

**Analogie**

C'est comme ranger ses courses avant de cuisiner. Tu pourrais chercher chaque ingrédient dans les sacs à chaque recette, mais si tu vas enchaîner 300 recettes, autant tout ranger proprement une fois au début.

In [23]:
train = xgb.DMatrix(X_train, label=y_train)
valid = xgb.DMatrix(X_val, label=y_val)

In [1]:
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("model", "xgboost")
        mlflow.log_params(params)
        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=1000,
            evals=[(valid, 'validation')],
            early_stopping_rounds=50
        )
        y_pred = booster.predict(valid)
        #rmse = mean_squared_error(y_val, y_pred, squared=False)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        mlflow.log_metric("rmse", rmse)

    return {'loss': rmse, 'status': STATUS_OK}

Ta compréhension est bonne dans les grandes lignes. Je précise les deux points que tu demandes et je corrige une nuance importante.

---

### `xgb.train` — comment fonctionne XGBoost

XGBoost est un algorithme de **boosting** : il construit des arbres de décision **les uns après les autres**, chaque nouvel arbre essayant de corriger les erreurs des précédents.

```
Arbre 1 → prédit grossièrement (ex: 12 min)
          erreur résiduelle = -3 min
Arbre 2 → apprend à prédire cette erreur
          erreur restante = +0.8 min
Arbre 3 → apprend à prédire ce qui reste
...
```

La prédiction finale = somme de tous les arbres. C'est très différent d'une régression linéaire qui apprend un seul jeu de coefficients d'un coup.

Les arguments :

| Argument | Rôle |
|---|---|
| `params` | Les hyperparamètres (profondeur des arbres, learning rate...) |
| `dtrain=train` | Les données d'entraînement au format DMatrix |
| `num_boost_round=1000` | Nombre **maximum** d'arbres à construire |
| `evals=[(valid, 'validation')]` | À chaque arbre, calcule l'erreur sur la validation |
| `early_stopping_rounds=50` | Arrête si l'erreur de validation ne s'améliore pas pendant 50 arbres d'affilée |

C'est là que le `DMatrix` de `valid` sert : XGBoost surveille automatiquement la validation à chaque itération sans que tu aies à intervenir.

**L'early stopping évite le surapprentissage.** Typiquement l'erreur de validation baisse, atteint un minimum vers l'arbre 300, puis remonte. XGBoost s'arrête à ce moment-là et garde le meilleur modèle, plutôt que d'aller jusqu'à 1000.

Ici `learning_rate` existe vraiment (contrairement au Lasso) : il contrôle à quel point chaque nouvel arbre corrige. Petit learning rate → corrections prudentes, il faut plus d'arbres mais c'est souvent plus précis.

---

### `fmin` — la recherche d'hyperparamètres

`fmin` vient de la librairie **hyperopt**. Son nom signifie "function minimize" : elle cherche les hyperparamètres qui **minimisent** la valeur retournée par ta fonction.

**Correction importante :** `fmin` ne cherche pas "le run où l'erreur est minimale" dans MLflow. Elle ne connaît même pas l'existence de MLflow. Elle est complètement indépendante — MLflow enregistre juste en parallèle, comme un observateur.

**Oui, `fn=objective` passe bien la fonction elle-même**, sans parenthèses. Tu ne l'appelles pas, tu la **donnes** à `fmin`, qui va l'appeler 50 fois avec des hyperparamètres différents.

Voici la boucle qui se déroule :

```
fmin tire un jeu d'hyperparamètres du search_space
   ↓
appelle objective(params)
   ↓
objective entraîne XGBoost, calcule le rmse, log dans MLflow
   ↓
objective retourne {'loss': rmse, 'status': STATUS_OK}
   ↓
fmin lit le 'loss' et en déduit où chercher ensuite
   ↓
(répète 50 fois)
```

C'est pour ça que le `return` à la fin de `objective` est obligatoire : c'est le **contrat** entre ta fonction et `fmin`. La clé doit s'appeler exactement `'loss'`, sinon hyperopt ne saura pas quoi minimiser.

| Argument | Rôle |
|---|---|
| `fn=objective` | La fonction à minimiser |
| `space=search_space` | Où chercher |
| `algo=tpe.suggest` | **Comment** chercher (voir ci-dessous) |
| `max_evals=50` | Nombre d'essais |
| `trials=Trials()` | Objet qui mémorise l'historique des essais |

---

### Le point le plus intéressant : `tpe.suggest`

C'est ce qui distingue hyperopt d'une simple grid search.

- **Grid search** : teste toutes les combinaisons d'une grille → explosion combinatoire
- **Random search** : tire au hasard → n'apprend rien des essais passés
- **TPE** (Tree-structured Parzen Estimator) : **apprend des essais précédents**

Le TPE construit un modèle probabiliste du genre *"les bons résultats avaient un learning_rate autour de 0.1 et un max_depth plutôt faible"*, puis concentre ses tirages suivants dans ces zones prometteuses. C'est de l'optimisation bayésienne.

D'où l'intérêt de l'objet `Trials()` : il garde l'historique dont le TPE a besoin pour raisonner.

---

### Détails sur le search_space

```python
'max_depth': scope.int(hp.quniform('max_depth', 4, 100, 1))
```
`quniform` tire un nombre entre 4 et 100 par pas de 1, mais retourne un **float** (ex: 7.0). Or `max_depth` doit être un entier → `scope.int` fait la conversion.

```python
'learning_rate': hp.loguniform('learning_rate', -3, 0)
```
⚠️ Piège : les bornes sont en **échelle logarithmique**. Donc ça tire entre e⁻³ et e⁰, soit environ **0.05 à 1**.

Pourquoi log ? Parce que pour un learning rate, passer de 0.001 à 0.01 est un changement aussi significatif que passer de 0.1 à 1. Une échelle linéaire passerait presque tout son temps dans les grandes valeurs.

```python
'objective': 'reg:linear'
```
⚠️ Attention à la confusion de vocabulaire ! Ici `objective` n'a **rien à voir** avec ta fonction Python `objective`. C'est un hyperparamètre XGBoost qui désigne la fonction de perte à optimiser (régression). Malheureux hasard de nommage dans le cours.

Par ailleurs `'reg:linear'` est **obsolète**. Chez toi il faut mettre :
```python
'objective': 'reg:squarederror'
```

Et comme d'habitude :
```python
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
```

##   Question : pourquoi la fonction de eprte est dans le search space = la fourchette d'hyperparamètres àoptimiser ?

**Ton intuition est juste, et il y a une vraie réponse.**

---

## D'abord, regarde bien le search_space

```python
search_space = {
    'max_depth': scope.int(hp.quniform(...)),      # ← varie
    'learning_rate': hp.loguniform(...),           # ← varie
    'reg_alpha': hp.loguniform(...),               # ← varie
    'reg_lambda': hp.loguniform(...),              # ← varie
    'min_child_weight': hp.loguniform(...),        # ← varie
    'objective': 'reg:linear',                     # ← constante !
    'seed': 42                                     # ← constante !
}
```

Les cinq premiers utilisent `hp.*` → ce sont des **distributions**, hyperopt va y piocher.
Les deux derniers sont des **valeurs fixes** → hyperopt les recopie telles quelles à chaque essai.

Donc `objective` n'est pas optimisé. Il ne varie jamais sur les 50 runs.

---

### Alors pourquoi est-il dans ce dictionnaire ?

Pour une raison purement **technique**. Regarde le chemin que prend ce dictionnaire :

```
search_space  →  fmin  →  objective(params)  →  xgb.train(params=params, ...)
```

`xgb.train` attend **un seul dictionnaire** contenant toute la configuration du modèle. L'API native de XGBoost ne fait aucune distinction entre "ce qui se règle" et "ce qui est structurel" — tout passe par `params`.

Donc `objective` et `seed` sont dans le search_space non pas parce qu'on veut les optimiser, mais parce que c'est le **seul véhicule** pour les faire arriver jusqu'à `xgb.train`.

C'est un peu bancal comme design, tu as raison de tiquer.

---

### Comparaison avec ce que tu connais

| | Choix de la loss | Réglage |
|---|---|---|
| **sklearn** | Encodé dans la **classe** : `LinearRegression`, `Lasso`, `Ridge` | Arguments du constructeur |
| **XGBoost natif** | Une clé du dict `params` | Les autres clés du même dict |

En sklearn, changer de fonction de perte = changer de classe. En XGBoost, c'est juste une chaîne de caractères, parce qu'un seul algorithme (le boosting d'arbres) sait gérer régression, classification et ranking. La loss est ce qui détermine laquelle des trois tu fais.

---

### Ceci dit, la nuance mérite d'être posée

`objective` **peut** légitimement être considéré comme un hyperparamètre au sens strict : tout ce qui est fixé avant l'entraînement et non appris à partir des données en est un.

Et en pratique, on le fait parfois varier ! Par exemple pour une régression :

```python
'reg:squarederror'      # pénalise fortement les grosses erreurs
'reg:absoluteerror'     # plus robuste aux valeurs aberrantes
'reg:pseudohubererror'  # compromis entre les deux
```

Sur tes durées de trajets, tu pourrais comparer ces trois-là et voir lequel donne le meilleur résultat. Ce serait un choix de modélisation légitime.

**Mais c'est un choix de nature différente** de celui du learning rate :
- `learning_rate` → question d'**optimisation** : à quelle vitesse converger
- `objective` → question de **modélisation** : qu'est-ce qu'une bonne prédiction, comment je veux pénaliser mes erreurs

Ta gêne vient de là, et elle est fondée. On mélange dans un même dictionnaire des paramètres qui n'ont pas le même statut conceptuel.

## autre question, pour quoi dans fmin, on écrit fn=objective et pas fn=objective(params) ou fn=objective(search_space) --> je pensais que la fonction objective devait avoir forcément un argument. Alors qu'ici on la passe sans argument dans la fonction fmin**


**Excellente question — c'est une distinction fondamentale en Python.**

La fonction a bien un argument. Mais ce n'est pas **toi** qui le fournis, c'est `fmin`.

---

### Avec ou sans parenthèses : deux choses très différentes

```python
objective          # ← la fonction ELLE-MÊME (un objet)
objective(params)  # ← le RÉSULTAT de son exécution
```

En Python, une fonction est un objet comme un autre. Tu peux la stocker dans une variable, la mettre dans une liste, ou la passer en argument — tant que tu ne mets pas de parenthèses.

```python
def dire_bonjour(nom):
    return f"Bonjour {nom}"

f = dire_bonjour           # f est maintenant la fonction
print(f)                   # <function dire_bonjour at 0x7f8b...>

f = dire_bonjour("Pierre") # f est maintenant le résultat
print(f)                   # "Bonjour Pierre"
```

---

### Ce qui se passerait avec `fn=objective(params)`

Deux problèmes en cascade :

**1. Ça planterait immédiatement**
```python
fmin(fn=objective(params), ...)
                    ↑
        NameError: 'params' is not defined
```
La variable `params` n'existe pas dans ton notebook. Elle n'existe qu'**à l'intérieur** de la fonction, le temps de son exécution.

**2. Même si elle existait, ce serait faux**

Python évaluerait `objective(params)` **avant** d'appeler `fmin`. Donc :
- L'entraînement se lancerait une fois, tout de suite
- `fmin` recevrait un dictionnaire `{'loss': 6.3, 'status': 'ok'}`
- `fmin` essaierait de l'appeler → `TypeError: 'dict' object is not callable`

---

### Qui fournit l'argument, alors ?

C'est `fmin` qui s'en charge, en interne. Schématiquement, voici ce qu'elle fait :

```python
# Pseudo-code simplifié de fmin
for i in range(max_evals):              # 50 fois
    params = tirer_du_space(search_space)   # fmin construit les params
    resultat = fn(params)                   # fmin APPELLE ta fonction
    memoriser(resultat['loss'])             # fmin lit la loss
```

Tu donnes la **recette**, `fmin` fournit les **ingrédients** et fait la cuisine 50 fois.

---

### Analogie

Imagine que tu embauches quelqu'un pour tester 50 gâteaux.

```python
# ✅ Tu lui donnes la recette
embaucher(recette=faire_gateau)

# ❌ Tu fais le gâteau toi-même et tu lui tends l'assiette
embaucher(recette=faire_gateau(farine))
```

Dans le deuxième cas, la personne reçoit un gâteau déjà fait. Elle ne peut plus en tester 50 variantes — elle n'a pas la recette.

---

### D'où vient le nom `params` alors ?

C'est juste le nom que **tu** as choisi pour le paramètre de ta fonction :

```python
def objective(params):   # ← ce nom est arbitraire
    ...
```

Tu aurais pu écrire `def objective(hyperparams)` ou `def objective(x)`, ça marcherait pareil. `fmin` ne connaît pas ce nom, elle passe juste sa valeur en premier argument.

La seule contrainte imposée par hyperopt est le **contrat** :
- ta fonction accepte **un** argument (le dict d'hyperparamètres)
- elle retourne un dict contenant la clé `'loss'`

In [24]:
search_space = {
    'max_depth': scope.int(hp.quniform('max_depth', 4, 20, 1)),
    'learning_rate': hp.loguniform('learning_rate', -3, 0),
    'reg_alpha': hp.loguniform('reg_alpha', -5, -1),
    'reg_lambda': hp.loguniform('reg_lambda', -6, -1),
    'min_child_weight': hp.loguniform('min_child_weight', -1, 3),
    #'objective': 'reg:linear',
    'objective': 'reg:squarederror',
    'seed': 42
}

#fmin vient de la librairie hyperopt. Son nom signifie "function minimize" : elle cherche les hyperparamètres qui minimisent la valeur retournée par ta fonction.

best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=10,
    trials=Trials()
)

[0]	validation-rmse:11.74945                          
[1]	validation-rmse:11.32128                          
[2]	validation-rmse:10.92616                          
[3]	validation-rmse:10.56171                          
[4]	validation-rmse:10.22659                          
[5]	validation-rmse:9.91868                           
[6]	validation-rmse:9.63566                           
[7]	validation-rmse:9.37667                           
[8]	validation-rmse:9.13976                           
[9]	validation-rmse:8.92310                           
[10]	validation-rmse:8.72526                          
[11]	validation-rmse:8.54438                          
[12]	validation-rmse:8.37898                          
[13]	validation-rmse:8.22870                          
[14]	validation-rmse:8.09128                          
[15]	validation-rmse:7.96725                          
[16]	validation-rmse:7.85441                          
[17]	validation-rmse:7.75221                          
[18]	valid

KeyboardInterrupt: 

In [25]:
mlflow.xgboost.autolog(disable=True)

Ci dessous, dans best_params, on prends les paramètre optimal qu'à doner la cellule du dessus. Wrokflow : 
- on a défini un espace de paramètres search_space
- on a défini la fonction objective 
- on a utilisé la méthode fmin de la librairie hyperopt : elle cherche les hyperparamètres (parmi le search space) qui minimisent la valeur retournée par l afonction "objective"
--> fmin fait plein de run, chaque run est loggé dans MLFlow (REMARQUE : comme ça prend 2h a touner littérallement, j'ai stoppé le run de la cellule)
--> en allant dans mlflow, on caompare tous les runs de la fonction fmin (on regarde les scatter plot, contour plot etc... et le temps qu'a duré le run)
--> après analyse, on se dit que le run XY a produit le meilleur modele
--> on prends le shypermaramètre de ce modele --> ce sont les best_params de la cellule ci-dessous : 

Par ailelurs, on a : 
- désactiver le autolog de mlflow pour éviter d'enregistré 2 fois le modele (cellule ci-dessus)(pas exactement compris pourquoi)
- enregistrer le preprocessing de la data avec 

```python
with open("models/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)
    mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")
```

- utiliser la méthode .log_model de mlflow (qui fonctionne avec le framework xgboost) pour enregistré le model, et plein de paramètres

Quatre bonnes questions. Je les prends dans l'ordre.

---

### 1. Pourquoi autolog enregistrerait le modèle deux fois

`mlflow.xgboost.autolog()` est un **interrupteur global**. Une fois activé, MLflow "surveille" tous les appels à `xgb.train()` et enregistre automatiquement, sans que tu demandes quoi que ce soit :

- les hyperparamètres
- les métriques d'entraînement à chaque itération
- **le modèle lui-même**
- les dépendances, la signature, un exemple d'input

Or dans cette cellule, tu enregistres déjà le modèle à la main :

```python
mlflow.xgboost.log_model(booster, artifact_path="models_mlflow")
```

Donc si autolog était actif, tu aurais **deux copies** du même modèle dans le même run :

```
artifacts/
├── model/            ← déposé automatiquement par autolog
└── models_mlflow/    ← déposé par ton log_model
```

Même modèle, deux dossiers, deux fois la place disque, et une confusion garantie quand tu voudras le recharger plus tard ("lequel je prends ?").

Pourquoi `disable=True` et pas simplement ne rien écrire ? Parce que dans une cellule précédente du notebook, autolog **a été activé** (souvent avant la partie hyperopt). Le réglage persiste tant que le kernel tourne. Il faut donc le désactiver explicitement.

---

### 2. Les lignes du preprocessor

```python
with open("models/preprocessor.b", "wb") as f_out:
    pickle.dump(dv, f_out)
mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")
```

**Ligne 1-2** — exactement le même mécanisme que le pickle qu'on a vu plus tôt : on sérialise l'objet `dv` (le DictVectorizer entraîné) dans un fichier binaire local, sur le disque de ta machine.

**Ligne 3** — `log_artifact` prend ce fichier local et le **copie dans le stockage d'artifacts de MLflow**, associé au run en cours. Les deux arguments :

| Argument | Rôle |
|---|---|
| `"models/preprocessor.b"` | Où trouver le fichier sur **ta machine** |
| `artifact_path="preprocessor"` | Dans quel sous-dossier le ranger **côté MLflow** |

Le second est juste de l'organisation. Sans lui, tout atterrit en vrac à la racine des artifacts du run.

**Pourquoi c'est indispensable ?**

Souviens-toi de la discussion sur `fit_transform` : le `dv` contient le **vocabulaire appris** — quelles colonnes existent et dans quel ordre. Le modèle XGBoost, lui, ne connaît que des numéros de colonnes.

```
données brutes → dv → matrice → booster → prédiction
                 ↑              ↑
            sans lui,      ne sait pas
          impossible de    interpréter
          construire la    autre chose
            matrice
```

Un modèle sans son préprocesseur est **inutilisable**. Les deux doivent voyager ensemble — c'est la même logique que le `pickle.dump((dv, lr), ...)` de tout à l'heure, sauf qu'ici on les stocke séparément parce que le modèle passe par `log_model`.

---

### 3. `log_model` et les frameworks

**Oui, tu as bien compris.** MLflow a des modules dédiés par framework, appelés **flavors** :

```python
mlflow.xgboost.log_model(...)
mlflow.sklearn.log_model(...)
mlflow.pytorch.log_model(...)
mlflow.tensorflow.log_model(...)
mlflow.lightgbm.log_model(...)
mlflow.statsmodels.log_model(...)
# et une trentaine d'autres
```

Chaque flavor sait comment sérialiser et recharger ce type précis de modèle.

**Qu'est-ce que ça apporte par rapport à un simple `log_artifact` du pickle ?** Beaucoup :

- Un fichier `MLmodel` décrivant le modèle (format, version, signature d'entrée/sortie)
- Un `requirements.txt` / `conda.yaml` avec les versions exactes des librairies → reproductibilité
- La possibilité de recharger avec `mlflow.xgboost.load_model(...)` sans savoir comment il a été sauvegardé
- **Le déploiement** : MLflow peut transformer ce dossier en API REST, en job Spark, en image Docker

Et si ton modèle n'a pas de flavor dédié (modèle maison, pipeline custom), il y a une porte de sortie universelle : `mlflow.pyfunc`, où tu écris toi-même la classe wrapper avec une méthode `predict`.

---

### 4. Une méthode pour tout enregistrer d'un coup ?

**Oui — et c'est précisément `autolog()`.** L'ironie est là : la ligne qu'on vient de désactiver fait exactement ça.

```python
mlflow.xgboost.autolog()   # tout est enregistré automatiquement
```

Le cours désactive autolog pour des raisons pédagogiques (voir ce qu'on enregistre, comprendre chaque brique) et pratiques :
- autolog logge le RMSE **d'entraînement** calculé par XGBoost, alors qu'ici on veut notre propre RMSE sur la validation
- on veut choisir les noms des artifacts
- on veut ajouter le preprocessor, qu'autolog ignore complètement

En mode manuel, tu peux quand même grouper par lot — remarque les pluriels :

```python
mlflow.log_params({'alpha': 0.1, 'max_depth': 10})    # plusieurs params d'un coup
mlflow.log_metrics({'rmse': 6.3, 'mae': 4.1})         # plusieurs métriques d'un coup
```

Mais il n'existe pas de fonction unique qui prendrait params + métriques + modèle ensemble. La philosophie de MLflow est : soit tu automatises tout (autolog), soit tu contrôles tout (manuel).

---

### 5. L'ordre des appels

**Techniquement, non, l'ordre n'a aucune importance.** Tout ce qui est dans le bloc `with` est rattaché au même run. Tu pourrais logger la métrique avant les params, ça donnerait le même résultat.

Il y a juste deux **contraintes de logique**, pas de MLflow :

- `log_artifact` doit venir **après** la création du fichier (sinon il n'existe pas encore)
- `log_model` doit venir **après** l'entraînement (sinon `booster` n'existe pas)

En revanche, une **bonne pratique** : logger les params **avant** l'entraînement. Si l'entraînement plante, le run se termine en statut `FAILED` mais tu gardes la trace de la configuration qui a échoué. C'est exactement ce que fait le code ici.

---

### Deux points d'attention pour ton exécution

```python
'objective': 'reg:linear',   # → remplace par 'reg:squarederror'
```

Et selon ta version de MLflow, `artifact_path=` dans `log_model` peut déclencher un warning de dépréciation au profit de `name=`. Si tu le vois :

```python
mlflow.xgboost.log_model(booster, name="models_mlflow")
```

In [41]:

#with mlflow.start_run(run_name="lasso-alpha-0.1")
with mlflow.start_run(run_name="test_name"):
    
    train = xgb.DMatrix(X_train, label=y_train)
    valid = xgb.DMatrix(X_val, label=y_val)

    best_params = {
        'learning_rate': 0.09585355369315604,
        'max_depth': 10, #30
        'min_child_weight': 1.060597050922164,
        'objective': 'reg:linear',
        'reg_alpha': 0.018060244040060163,
        'reg_lambda': 0.011658731377413597,
        'seed': 42
    }

    mlflow.log_params(best_params)

    booster = xgb.train(
        params=best_params,
        dtrain=train,
        num_boost_round=1000,
        evals=[(valid, 'validation')],
        early_stopping_rounds=50
    )

    y_pred = booster.predict(valid)
    #rmse = mean_squared_error(y_val, y_pred, squared=False)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mlflow.log_metric("rmse", rmse)

    with open("models/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)
    mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

    mlflow.xgboost.log_model(booster, artifact_path="models_mlflow")

/home/codespace/miniconda3/envs/mlopszoomcamp/lib/python3.11/site-packages/xgboost/callback.py:385: UserWarning: [20:08:46] WARNING: /__w/xgboost/xgboost/src/objective/regression_obj.cu:277: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()


[0]	validation-rmse:11.45836
[1]	validation-rmse:10.79725
[2]	validation-rmse:10.22104
[3]	validation-rmse:9.71995
[4]	validation-rmse:9.28684
[5]	validation-rmse:8.91201
[6]	validation-rmse:8.59100
[7]	validation-rmse:8.31518
[8]	validation-rmse:8.07799
[9]	validation-rmse:7.87706
[10]	validation-rmse:7.70358
[11]	validation-rmse:7.55680
[12]	validation-rmse:7.43295
[13]	validation-rmse:7.32592
[14]	validation-rmse:7.23546
[15]	validation-rmse:7.15960
[16]	validation-rmse:7.09257
[17]	validation-rmse:7.03714
[18]	validation-rmse:6.98973
[19]	validation-rmse:6.94875
[20]	validation-rmse:6.91366
[21]	validation-rmse:6.88229
[22]	validation-rmse:6.85548
[23]	validation-rmse:6.83337
[24]	validation-rmse:6.81390
[25]	validation-rmse:6.79685
[26]	validation-rmse:6.78236
[27]	validation-rmse:6.76942
[28]	validation-rmse:6.75775
[29]	validation-rmse:6.74729
[30]	validation-rmse:6.73853
[31]	validation-rmse:6.73054
[32]	validation-rmse:6.72400
[33]	validation-rmse:6.71730
[34]	validation-rmse:

2026/09/13 20:10:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


In [23]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.svm import LinearSVR

mlflow.sklearn.autolog()

for model_class in (RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, LinearSVR):

    with mlflow.start_run():

        mlflow.log_param("train-data-path", "./data/green_tripdata_2021-01.csv")
        mlflow.log_param("valid-data-path", "./data/green_tripdata_2021-02.csv")
        mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

        mlmodel = model_class()
        mlmodel.fit(X_train, y_train)

        y_pred = mlmodel.predict(X_val)
        rmse = mean_squared_error(y_val, y_pred, squared=False)
        mlflow.log_metric("rmse", rmse)
        

/Users/cristian.martinez/miniconda3/envs/exp-tracking-env/lib/python3.9/site-packages/sklearn/svm/_base.py:1206: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
